In [ ]:
#Imports
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

df = pd.read_csv("data/walmart_sales_engineered.csv")

In [2]:
##Sorting##
df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(["Date", "Store"]).reset_index(drop=True)

In [3]:
#Train/Test Split
# Make sure Date is datetime first
df["Date"] = pd.to_datetime(df["Date"])

# Define cutoff (you choose based on your dataset)
cutoff_date = df["Date"].quantile(0.8)

train = df[df["Date"] <= cutoff_date].copy()
test = df[df["Date"] > cutoff_date].copy()

y_train = train["Weekly_Sales"]
y_test = test["Weekly_Sales"]

In [13]:
#Baseline###########################################################################################
baseline_pred = test["Sales_Lag_1"]

baseline_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": baseline_pred.values
})

baseline_df.to_csv("results/baseline_predictions.csv", index=False)

mae = mean_absolute_error(y_test, baseline_pred)
rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

print("Baseline")
print(mae)
print(rmse)

Baseline
50864.95688888889
76052.9862594463


In [5]:
#Basic Linear Regression
basic_features = [
    "Store",
    "Holiday_Flag",
    "Temperature",
    "Fuel_Price",
    "CPI",
    "Unemployment",
    "Year",
    "Month",
    "Quarter",
    "Week",
    "DayOfWeek",
    "Season_Fall",
    "Season_Spring",
    "Season_Summer",
    "Season_Winter"
]

#Create Data
X_basic_train = train[basic_features]
X_basic_test = test[basic_features]

#Train
basic_model = LinearRegression()

basic_model.fit(X_basic_train, y_train)

joblib.dump(
    basic_model,
    "models/linear_regression_basic.pkl"
)

#Predict
basic_predictions = basic_model.predict(X_basic_test)

#Evaluate
basic_mae = mean_absolute_error(y_test, basic_predictions)
basic_rmse = np.sqrt(mean_squared_error(y_test, basic_predictions))
basic_r2 = r2_score(y_test, basic_predictions)

print("Basic Linear Regression")
print(basic_mae)
print(basic_rmse)
print(basic_r2)

Basic Linear Regression
417432.7435536207
495604.6968130665
0.1423327619457948


In [6]:
#Enhanced Linear Regression
enhanced_features = [
    "Store",
    "Holiday_Flag",
    "Temperature",
    "Fuel_Price",
    "CPI",
    "Unemployment",
    "Year",
    "Month",
    "Quarter",
    "Week",
    "DayOfWeek",
    "Sales_Lag_1",
    "Sales_Lag_4",
    "Rolling_Mean_4",
    "Rolling_Mean_12",
    "Rolling_STD_4",
    "Season_Fall",
    "Season_Spring",
    "Season_Summer",
    "Season_Winter"
]

#Training
X_enhanced_train = train[enhanced_features]
X_enhanced_test = test[enhanced_features]

enhanced_model = LinearRegression()

enhanced_model.fit(X_enhanced_train, y_train)

joblib.dump(
    enhanced_model,
    "models/linear_regression_enhanced.pkl"
)

#Predict
enhanced_predictions = enhanced_model.predict(X_enhanced_test)

#Evaluate
enhanced_mae = mean_absolute_error(y_test, enhanced_predictions)
enhanced_rmse = np.sqrt(mean_squared_error(y_test, enhanced_predictions))
enhanced_r2 = r2_score(y_test, enhanced_predictions)

print("Enhanced Linear Regression")
print(enhanced_mae)
print(enhanced_rmse)
print(enhanced_r2)

predictions_df = pd.DataFrame({
    "Date": test["Date"],
    "Store": test["Store"],
    "Actual": y_test.values,
    "Predicted": enhanced_predictions
})

predictions_df.to_csv(
    "results/linear_regression_predictions.csv",
    index=False
)

coefficients = pd.DataFrame({
    "Feature": enhanced_features,
    "Coefficient": enhanced_model.coef_
})

coefficients.to_csv(
    "results/feature_coefficients.csv",
    index=False
)

Enhanced Linear Regression
53457.32529535927
73172.7030932309
0.981304107972177


In [7]:
#Savie Results
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Linear Regression (Basic)",
        "Linear Regression (Enhanced)"
    ],
    "MAE": [
        mae,
        basic_mae,
        enhanced_mae
    ],
    "RMSE": [
        rmse,
        basic_rmse,
        enhanced_rmse
    ],
    "R²": [
        np.nan,
        basic_r2,
        enhanced_r2
    ]
})

results

results.to_csv("results/model_metrics.csv", index=False)

In [9]:
#Random Forest Model###########################################################

#Create Model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

#Train
rf_model.fit(X_enhanced_train, y_train)

#Predict
rf_predictions = rf_model.predict(X_enhanced_test)

#Evaluate
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("RANDOM FOREST RESULTS")
print("---------------------")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R²:", rf_r2)

#results
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Linear Regression (Basic)",
        "Linear Regression (Enhanced)",
        "Random Forest"
    ],
    "MAE": [
        mae,
        basic_mae,
        enhanced_mae,
        rf_mae
    ],
    "RMSE": [
        rmse,
        basic_rmse,
        enhanced_rmse,
        rf_rmse
    ],
    "R²": [
        np.nan,
        basic_r2,
        enhanced_r2,
        rf_r2
    ]
})

results

joblib.dump(
    rf_model,
    "models/random_forest.pkl"
)

rf_predictions_df = pd.DataFrame({
    "Date": test["Date"],
    "Store": test["Store"],
    "Actual": y_test.values,
    "Predicted": rf_predictions
})

rf_predictions_df.to_csv(
    "results/random_forest_predictions.csv",
    index=False
)

RANDOM FOREST RESULTS
---------------------
MAE: 42522.134997094
RMSE: 61186.08731585186
R²: 0.9869276593038431


In [12]:
#XGBoost Model############################################

#Create Model
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

#Train Model
xgb_model.fit(X_enhanced_train, y_train)

#Predict
xgb_predictions = xgb_model.predict(X_enhanced_test)

#Evaluate
xgb_mae = mean_absolute_error(y_test, xgb_predictions)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_predictions))
xgb_r2 = r2_score(y_test, xgb_predictions)

print("XGBOOST RESULTS")
print("----------------")
print("MAE:", xgb_mae)
print("RMSE:", xgb_rmse)
print("R²:", xgb_r2)

#Results
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Linear Regression (Basic)",
        "Linear Regression (Enhanced)",
        "Random Forest",
        "XGBoost"
    ],
    "MAE": [
        mae,
        basic_mae,
        enhanced_mae,
        rf_mae,
        xgb_mae
    ],
    "RMSE": [
        rmse,
        basic_rmse,
        enhanced_rmse,
        rf_rmse,
        xgb_rmse
    ],
    "R²": [
        np.nan,
        basic_r2,
        enhanced_r2,
        rf_r2,
        xgb_r2
    ]
})

results

joblib.dump(
    xgb_model,
    "models/xgboost.pkl"
)

xgb_predictions_df = pd.DataFrame({
    "Date": test["Date"],
    "Store": test["Store"],
    "Actual": y_test.values,
    "Predicted": xgb_predictions
})

xgb_predictions_df.to_csv(
    "results/xgboost_predictions.csv",
    index=False
)

XGBOOST RESULTS
----------------
MAE: 36897.5222724359
RMSE: 52839.25824265576
R²: 0.9902509697368989
